# Module 3.0 — Prompting Baselines: Zero- and Few-Shot
**DeskMate SLM Curriculum · Phase 3 · Notebook 15**

Run on **Colab CPU** — no GPU needed.

By the end you will have:
- Zero-shot and few-shot accuracy / macro-F1 on the gold set
- Per-query latency and cost estimates
- A fine-tune decision note for DeskMate
- `reports/baseline_report.md`

**API paths (auto-detected):**
- Anthropic Haiku via `ANTHROPIC_API_KEY` in Colab Secrets
- Ollama `qwen2.5:1.5b` if running locally
- Mock (deterministic stub) — works everywhere, no key needed

Read `3.0_prompting_baselines.md` for full theory.

---

## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q anthropic==0.34.0 scikit-learn==1.5.1

In [ ]:
import random, json, pathlib, time, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print('Libraries loaded.')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC = PROJECT_ROOT / 'data' / 'processed'
REPORTS   = PROJECT_ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print(f'Runtime : {RUNTIME}')

## Step 2 — Label Maps

In [ ]:
maps_path = DATA_PROC / 'label_maps.json'
if maps_path.exists():
    lm = json.loads(maps_path.read_text())
    INTENT2ID = lm['INTENT2ID']
    ID2INTENT = {int(k): v for k, v in lm['ID2INTENT'].items()}
else:
    INTENTS   = ['account_access','account_settings','billing_dispute','billing_inquiry',
                 'cancellation','data_privacy','feature_request','onboarding',
                 'outage_report','payment_failure','performance_issue','refund_request',
                 'technical_bug','usage_question','out_of_scope']
    INTENT2ID = {i: n for n, i in enumerate(INTENTS)}
    ID2INTENT = {n: i for n, i in enumerate(INTENTS)}

INTENTS = list(INTENT2ID.keys())
N_INTENTS = len(INTENTS)
print(f'Intent classes: {N_INTENTS}')

## Step 3 — Load Gold Set

In [ ]:
def make_gold_placeholder(n_per_class=5):
    prios = ['low','medium','high']
    templates = [
        'I have a problem with {intent_words} and need help.',
        'Please assist me with my {intent_words} issue.',
        'Urgent: {intent_words} is not working for me.',
        'I cannot complete the {intent_words} process.',
        'Something is wrong with {intent_words} on my account.',
    ]
    rows = []
    for intent in INTENTS:
        iw = intent.replace('_', ' ')
        for i in range(n_per_class):
            rows.append({'text': templates[i].format(intent_words=iw),
                         'intent': intent, 'priority': prios[i % 3]})
    return rows

gold_path = DATA_PROC / 'gold.jsonl'
if not gold_path.exists():
    print('gold.jsonl not found — using placeholder (run Module 2.1 first)')
    DATA_PROC.mkdir(parents=True, exist_ok=True)
    rows = make_gold_placeholder(5)
    gold_path.write_text('\n'.join(json.dumps(r) for r in rows))

gold_df = pd.read_json(gold_path, lines=True)
print(f'Gold set: {len(gold_df)} examples across {gold_df["intent"].nunique()} intents')
print(gold_df['intent'].value_counts().to_string())

## Step 4 — Build Few-Shot Preamble

One representative example per intent class drawn from the training set.
**Never use gold set examples as few-shot seeds.**

In [ ]:
# Canonical one-per-class examples (hard-coded to avoid data leakage from gold set)
FEW_SHOT_EXAMPLES = [
    ('account_access',     'I cannot log in. My password reset email never arrived.'),
    ('account_settings',   'How do I change my notification preferences and timezone?'),
    ('billing_dispute',    'I was charged twice for the Pro plan last month.'),
    ('billing_inquiry',    'What is included in the Enterprise tier pricing?'),
    ('cancellation',       'I would like to cancel my subscription immediately.'),
    ('data_privacy',       'Please delete all my personal data under GDPR Article 17.'),
    ('feature_request',    'It would be great to have a dark mode in the dashboard.'),
    ('onboarding',         'I just signed up — how do I connect my first data source?'),
    ('outage_report',      'Your service has been down for 2 hours. Our team is blocked.'),
    ('payment_failure',    'My credit card payment failed but I was not refunded.'),
    ('performance_issue',  'The dashboard takes over 30 seconds to load every time.'),
    ('refund_request',     'I was billed after cancelling. Please issue a refund.'),
    ('technical_bug',      'The CSV export button throws a 500 error on every browser.'),
    ('usage_question',     'How do I create a custom report with date range filters?'),
    ('out_of_scope',       'What is the weather like in San Francisco today?'),
]

# Build preamble string without triple quotes
preamble_lines = ['Examples:']
for intent, text in FEW_SHOT_EXAMPLES:
    preamble_lines.append(f'Ticket: "{text}"')
    preamble_lines.append(f'Intent: {intent}')
    preamble_lines.append('')

FEW_SHOT_PREAMBLE = '\n'.join(preamble_lines)
print(FEW_SHOT_PREAMBLE[:400])
print(f'... ({len(FEW_SHOT_PREAMBLE)} chars total, ~{len(FEW_SHOT_PREAMBLE)//4} tokens)')

## Step 5 — Prompt Templates

In [ ]:
INTENT_LIST = ', '.join(INTENTS)

def zero_shot_prompt(text):
    return (
        'You are a support ticket classifier. Read the ticket and output ONLY the intent label.\n'
        'Valid intents: ' + INTENT_LIST + '\n\n'
        'Ticket: ' + text + '\n'
        'Intent:'
    )

def few_shot_prompt(text):
    return (
        'You are a support ticket classifier. Output ONLY the intent label.\n\n'
        + FEW_SHOT_PREAMBLE
        + 'Ticket: ' + text + '\n'
        'Intent:'
    )

# Preview
sample = 'I cannot access my account after the password reset.'
print('=== Zero-shot prompt ===')
print(zero_shot_prompt(sample))
print()
print('=== Few-shot prompt (last 3 lines) ===')
fp = few_shot_prompt(sample)
print('\n'.join(fp.splitlines()[-3:]))

## Step 6 — API Client (Anthropic / Ollama / Mock)

Auto-detects available path. Mock is deterministic (seeded) so results are reproducible.

In [ ]:
import os

API_PATH = 'mock'
anthropic_client = None

# Try Anthropic
try:
    import anthropic as _anthropic
    _key = None
    if RUNTIME == 'colab':
        try:
            from google.colab import userdata
            _key = userdata.get('ANTHROPIC_API_KEY')
        except Exception:
            pass
    _key = _key or os.environ.get('ANTHROPIC_API_KEY')
    if _key:
        anthropic_client = _anthropic.Anthropic(api_key=_key)
        API_PATH = 'anthropic'
except ImportError:
    pass

# Try Ollama
if API_PATH == 'mock':
    try:
        import urllib.request
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        API_PATH = 'ollama'
    except Exception:
        pass

print(f'API path: {API_PATH}')
if API_PATH == 'mock':
    print('  Running in mock mode — set ANTHROPIC_API_KEY for real results.')

In [ ]:
def call_model(prompt, model_path=API_PATH):
    if model_path == 'anthropic':
        msg = anthropic_client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=20,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return msg.content[0].text.strip(), msg.usage.input_tokens, msg.usage.output_tokens

    elif model_path == 'ollama':
        import urllib.request, urllib.parse
        payload = json.dumps({'model': 'qwen2.5:1.5b', 'prompt': prompt,
                              'stream': False, 'options': {'num_predict': 20}}).encode()
        req = urllib.request.Request(
            'http://localhost:11434/api/generate',
            data=payload, headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=30) as r:
            data = json.loads(r.read())
        return data['response'].strip(), len(prompt)//4, 5

    else:   # mock
        # Seeded pseudo-random: deterministic but not trivially correct
        h = hash(prompt) % (2**32)
        rng = random.Random(h)
        # 60% chance of picking the right intent (heuristic from prompt content)
        for intent in INTENTS:
            if intent.replace('_', ' ') in prompt.lower():
                if rng.random() < 0.60:
                    return intent, len(prompt)//4, 3
        return rng.choice(INTENTS), len(prompt)//4, 3

def parse_intent(raw):
    raw = raw.strip().lower().rstrip('.,;:')
    for prefix in ('intent:', 'label:', 'answer:', 'classification:'):
        if raw.startswith(prefix):
            raw = raw[len(prefix):].strip()
    for intent in INTENTS:
        if intent in raw:
            return intent
    return 'out_of_scope'

## Step 7 — Zero-Shot Baseline

In [ ]:
def run_baseline(prompt_fn, label='baseline', sample_n=None):
    df = gold_df.copy()
    if sample_n:
        df = df.sample(min(sample_n, len(df)), random_state=SEED)
    preds, latencies, in_toks, out_toks = [], [], [], []
    for _, row in df.iterrows():
        prompt = prompt_fn(row['text'])
        t0 = time.perf_counter()
        raw, n_in, n_out = call_model(prompt)
        latencies.append((time.perf_counter() - t0) * 1000)
        preds.append(parse_intent(raw))
        in_toks.append(n_in); out_toks.append(n_out)
    y_true  = [INTENT2ID.get(i, 0) for i in df['intent']]
    y_pred  = [INTENT2ID.get(p, 0) for p in preds]
    acc     = accuracy_score(y_true, y_pred)
    f1      = f1_score(y_true, y_pred, average='macro', zero_division=0)
    mean_ms = np.mean(latencies)
    tot_in  = sum(in_toks); tot_out = sum(out_toks)
    return {
        'label': label, 'n': len(df),
        'accuracy': acc, 'f1': f1,
        'mean_latency_ms': mean_ms,
        'total_input_tokens': tot_in, 'total_output_tokens': tot_out,
        'preds': preds, 'true': df['intent'].tolist(),
    }

print('Running zero-shot baseline ...')
zs = run_baseline(zero_shot_prompt, label='zero-shot')
print(f'Zero-shot: accuracy={zs["accuracy"]*100:.1f}%  macro-F1={zs["f1"]*100:.1f}%  '
      f'latency={zs["mean_latency_ms"]:.0f}ms/query')

## Step 8 — Few-Shot Baseline

In [ ]:
print('Running few-shot baseline ...')
fs = run_baseline(few_shot_prompt, label='few-shot')
print(f'Few-shot:  accuracy={fs["accuracy"]*100:.1f}%  macro-F1={fs["f1"]*100:.1f}%  '
      f'latency={fs["mean_latency_ms"]:.0f}ms/query')

In [ ]:
# Comparison table
print(f'\n{"":-<65}')
print(f'{"":<14} {"Accuracy":>10} {"Macro-F1":>10} {"Latency":>12} {"Input tok":>10}')
print(f'{"":-<65}')
for r in [zs, fs]:
    print(f'{r["label"]:<14} {r["accuracy"]*100:>9.1f}% {r["f1"]*100:>9.1f}% '
          f'{r["mean_latency_ms"]:>10.0f}ms {r["total_input_tokens"]/r["n"]:>10.0f}')
print(f'{"":-<65}')

## Step 9 — Error Analysis (Few-Shot)

In [ ]:
from collections import Counter

errors = [(t, p) for t, p in zip(fs['true'], fs['preds']) if t != p]
print(f'Errors: {len(errors)} / {fs["n"]} ({len(errors)/fs["n"]*100:.1f}%)')
print()
confusion_pairs = Counter(errors).most_common(10)
print('Top confusion pairs (true → predicted):')
for (true_l, pred_l), cnt in confusion_pairs:
    print(f'  {true_l:<25} → {pred_l:<25} x{cnt}')

In [ ]:
# Per-class F1 chart
y_true_fs = [INTENT2ID.get(i, 0) for i in fs['true']]
y_pred_fs = [INTENT2ID.get(p, 0) for p in fs['preds']]
per_class = f1_score(y_true_fs, y_pred_fs, average=None, zero_division=0)

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#DD8452' if f < 0.5 else '#4C72B0' for f in per_class]
ax.bar(range(N_INTENTS), per_class, color=colors)
ax.axhline(np.mean(per_class), color='black', linestyle='--',
           label=f'Macro-F1 = {np.mean(per_class)*100:.1f}%')
ax.set_xticks(range(N_INTENTS))
ax.set_xticklabels([ID2INTENT[i][:12] for i in range(N_INTENTS)],
                   rotation=45, ha='right', fontsize=8)
ax.set_ylabel('F1')
ax.set_title('Per-Class F1 — Few-Shot Baseline')
ax.legend()
plt.tight_layout()
plt.savefig(str(REPORTS / 'baseline_per_class_f1.png'), bbox_inches='tight')
plt.show()
print('Orange bars = F1 < 0.5 (intents the base model struggles with)')

## Step 10 — Cost Projection

In [ ]:
# Anthropic Haiku pricing (as of knowledge cutoff)
PRICE_IN_PER_MTok  = 0.25   # USD per million input tokens
PRICE_OUT_PER_MTok = 1.25   # USD per million output tokens

avg_in  = fs['total_input_tokens'] / fs['n']
avg_out = fs['total_output_tokens'] / fs['n']
cost_per_q = avg_in / 1e6 * PRICE_IN_PER_MTok + avg_out / 1e6 * PRICE_OUT_PER_MTok

print(f'Few-shot prompt: avg {avg_in:.0f} input + {avg_out:.0f} output tokens/query')
print(f'Cost per query : ${cost_per_q:.6f}')
print()
print(f'{"Queries/day":<20} {"Cost/day":>12} {"Cost/month":>14} {"Cost/year":>12}')
print('-' * 62)
for qpd in [100, 1_000, 10_000, 100_000]:
    daily   = qpd * cost_per_q
    monthly = daily * 30
    yearly  = daily * 365
    print(f'{qpd:<20,} ${daily:>11.2f} ${monthly:>13.2f} ${yearly:>11.2f}')
print()
print('Compare: self-hosted MiniLM (Module 2.4) = $0/query after one-time fine-tune.')

## Step 11 — Fine-Tune Decision

In [ ]:
zs_f1 = zs['f1'] * 100
fs_f1 = fs['f1'] * 100

# Encoder classifier result (from Module 2.4 / 2.6 if available)
encoder_report = REPORTS / 'eval_report.md'
encoder_f1 = None
if encoder_report.exists():
    for line in encoder_report.read_text().splitlines():
        if 'Macro-F1' in line and ':' in line:
            try:
                encoder_f1 = float(line.split(':')[1].strip().rstrip('%'))
            except ValueError:
                pass
            break

TARGET_F1 = 85.0

print('=== DeskMate Fine-Tune Decision ===')
print()
print(f'Zero-shot baseline   : {zs_f1:.1f}% macro-F1')
print(f'Few-shot baseline    : {fs_f1:.1f}% macro-F1')
if encoder_f1:
    print(f'Fine-tuned encoder   : {encoder_f1:.1f}% macro-F1  (Module 2.4)')
print(f'Production target    : {TARGET_F1:.0f}% macro-F1')
print()
gap = TARGET_F1 - fs_f1
if gap <= 0:
    verdict = 'PROMPT. The few-shot baseline already meets the target. Fine-tuning is premature.'
elif gap <= 5:
    verdict = ('MARGINAL. Gap is only {:.1f}pp. Fine-tune only if cost or latency at '
               'scale is a constraint.').format(gap)
elif gap <= 15:
    verdict = ('FINE-TUNE. {:.1f}pp gap is operationally significant. '
               'The encoder classifier (Module 2.4) is the right choice: '
               'faster, cheaper, and higher accuracy.').format(gap)
else:
    verdict = ('FINE-TUNE URGENTLY. {:.1f}pp gap means {:.0f}% of tickets are '
               'mis-routed. Baseline is unacceptable for production.').format(
               gap, 100 - fs_f1)

print(f'Gap to target: {gap:.1f}pp')
print(f'Verdict: {verdict}')

## Step 12 — Write Baseline Report

In [ ]:
lines = [
    '# DeskMate — Prompting Baseline Report\n',
    '## Summary\n',
    f'- Gold set size  : {zs["n"]}',
    f'- API path       : {API_PATH}',
    '',
    '## Results\n',
    f'| Strategy   | Accuracy | Macro-F1 | Latency    | Tok/query |',
    f'|---|---|---|---|---|',
]
for r in [zs, fs]:
    lines.append(
        f'| {r["label"]:<10} | {r["accuracy"]*100:>7.1f}% | {r["f1"]*100:>7.1f}% '
        f'| {r["mean_latency_ms"]:>8.0f}ms | {r["total_input_tokens"]/r["n"]:>9.0f} |'
    )
lines += [
    '',
    '## Cost Projection (Haiku, few-shot)\n',
    f'- Per query: ${cost_per_q:.6f}',
    f'- 1k queries/day  → ${1000*cost_per_q:.2f}/day',
    f'- 10k queries/day → ${10000*cost_per_q:.2f}/day',
    '',
    '## Top Confusion Pairs (few-shot)\n',
]
for (true_l, pred_l), cnt in confusion_pairs[:5]:
    lines.append(f'- {true_l} → {pred_l}: {cnt}x')
lines += [
    '',
    '## Fine-Tune Decision\n',
    f'- Target macro-F1: {TARGET_F1:.0f}%',
    f'- Gap from few-shot baseline: {gap:.1f}pp',
    f'- **{verdict}**',
]

report_path = REPORTS / 'baseline_report.md'
report_path.write_text('\n'.join(lines))
print(f'Report written: {report_path}')
print()
print('\n'.join(lines))

## Checkpoint

> **Your few-shot baseline hits 72% accuracy. How do you decide whether fine-tuning is worth the effort to reach 88%?**

Write your answer below. Cover:
1. Whether the 16pp gap is operationally acceptable.
2. Data availability for SFT.
3. Scale / cost argument.
4. Error type (random vs systematic).

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| Baseline report | `reports/baseline_report.md` |
| Per-class F1 chart | `reports/baseline_per_class_f1.png` |

**What you've built:** a rigorous prompting baseline that quantifies what the base model
can do without any fine-tuning — and a data-driven fine-tune decision for DeskMate.

**Next:** Module 3.1 — Choosing a base model for the decoder SLM.
Compare Qwen2.5-1.5B, SmolLM2-1.7B, and Phi-3-mini on license, context length,
benchmark scores, and DeskMate fit.